# Les directions qui ne tournent pas

| Module | Durée | Prérequis |
|---|---|---|
| M7 – Valeurs propres et dynamique | 75 min | M5 |

In [ ]:
# Style graphique Liouaï : couleurs du logo, axes sobres
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

LIOUAI = {"navy": "#243743", "vert": "#28B78D", "vert_texte": "#167A5C", "gris": "#A1A9AE",
          "gris_texte": "#5C6870", "jaune": "#E5C800", "fond": "#F3F5F6"}
plt.rcParams.update({
    "axes.prop_cycle": plt.cycler(color=[LIOUAI["vert"], LIOUAI["navy"], LIOUAI["gris"],
                                         LIOUAI["jaune"], LIOUAI["vert_texte"], LIOUAI["gris_texte"]]),
    "axes.edgecolor": LIOUAI["gris"], "axes.labelcolor": LIOUAI["navy"],
    "axes.titlecolor": LIOUAI["navy"], "axes.titleweight": "bold",
    "axes.spines.top": False, "axes.spines.right": False,
    "xtick.color": LIOUAI["gris_texte"], "ytick.color": LIOUAI["gris_texte"],
    "text.color": LIOUAI["navy"], "grid.color": "#E1E6E9", "legend.frameon": False,
    "font.family": "sans-serif", "font.sans-serif": ["Roboto", "Helvetica", "Arial", "DejaVu Sans"],
    "figure.dpi": 100,
})
# Dégradé de marque : gris clair -> vert -> navy (et sa version inversée)
if "liouai" not in mpl.colormaps:
    _cmap = LinearSegmentedColormap.from_list("liouai", [LIOUAI["fond"], LIOUAI["vert"], LIOUAI["navy"]])
    mpl.colormaps.register(_cmap)
    mpl.colormaps.register(_cmap.reversed())

## ① Définir la question

Un fabricant de vêtements de sport teste un nouveau tissu élastique. La machine d'essai saisit un échantillon et l'étire. Avant l'essai, on dessine sur le tissu un petit cercle ; après l'étirement, le cercle est devenu une **ellipse**, un peu inclinée.

Les mesures montrent que la déformation est linéaire (module M3) : un point du tissu repéré par le vecteur $\mathbf{u}$ arrive en $A\mathbf{u}$, avec

$$
A = \begin{pmatrix} 1{,}6 & 0{,}3 \\ 0{,}3 & 1{,}1 \end{pmatrix}.
$$

Presque tous les traits dessinés sur le tissu **tournent** pendant l'essai. Pour couper les pièces d'un maillot, la styliste veut connaître les directions du tissu qui sont **seulement étirées, sans tourner**, et de combien.

**La question : existe-t-il des directions qu'une transformation linéaire ne fait pas tourner ? Comment les trouver, et que nous apprennent-elles ?**

**Activité** : D'après le dessin d'un cercle devenu ellipse, où chercheriez-vous ces directions ? Combien vous attendez-vous à en trouver ?

Indiquez vos réponses et justifiez-les.

## ② Estimer avant de calculer

In [ ]:
pari_angle_direction_principale = 45     # angle avec l'horizontale, en degrés (modifiez)
pari_etirement_maximal = 1.6             # facteur d'étirement dans cette direction (modifiez)

## ③ Abstraire : vecteurs propres et valeurs propres

Un vecteur non nul $\mathbf{v}$ est un **vecteur propre** de $A$ s'il garde sa direction : son image lui est proportionnelle.

$$
A\mathbf{v} = \lambda\,\mathbf{v}
$$

Le nombre $\lambda$ est la **valeur propre** associée : le facteur d'étirement dans cette direction. Si $\lambda < 0$, la direction est conservée mais le sens est retourné ; si $\lambda = 0$, la direction est écrasée.

![Schéma : la matrice A transforme le cercle unité en ellipse ; un vecteur quelconque u tourne, les vecteurs propres v1 et v2 restent sur leur droite et sont étirés par 3 et par 1](../../assets/schemas/schema-vecteurs-propres.svg)

**Comment les trouver à la main ?** L'égalité s'écrit $(A - \lambda I)\mathbf{v} = \mathbf{0}$ avec $\mathbf{v} \neq \mathbf{0}$ : la matrice $A - \lambda I$ écrase une direction, elle n'est donc pas inversible (module M4). Son déterminant est nul :

$$
\det(A - \lambda I) = 0
\qquad\Longleftrightarrow\qquad
\lambda^2 - \operatorname{tr}(A)\,\lambda + \det A = 0 \quad \text{pour une matrice } 2 \times 2,
$$

où la **trace** $\operatorname{tr}(A)$ est la somme des coefficients diagonaux. C'est le **polynôme caractéristique** : ses racines sont les valeurs propres.

**Activité** : Pour la matrice du schéma, $A = \begin{pmatrix} 2 & 1 \\ 1 & 2 \end{pmatrix}$, écrivez le polynôme caractéristique, trouvez $\lambda_1 = 3$ et $\lambda_2 = 1$, puis un vecteur propre pour chacune en résolvant $(A - \lambda I)\mathbf{v} = \mathbf{0}$.

Indiquez vos réponses et justifiez-les.

## ④ Calculer

### Explorer : de combien chaque direction tourne-t-elle ?

On fait tourner un vecteur unitaire $\mathbf{u}$ d'un demi-tour, et l'on mesure l'angle entre $\mathbf{u}$ et son image $A\mathbf{u}$. Les directions propres sont celles où cet angle s'annule.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

A = np.array([[1.6, 0.3], [0.3, 1.1]])

def angle_signe(u, w):
    # angle (en degrés) dont il faut tourner u pour obtenir la direction de w
    return np.degrees(np.arctan2(u[0] * w[1] - u[1] * w[0], u @ w))

thetas = np.linspace(0, 180, 721)
rotations = [angle_signe(np.array([np.cos(t), np.sin(t)]), A @ np.array([np.cos(t), np.sin(t)]))
             for t in np.radians(thetas)]

fig, ax = plt.subplots(figsize=(8, 3.6))
ax.plot(thetas, rotations, color=LIOUAI["navy"])
ax.axhline(0, color=LIOUAI["gris"])
ax.set_xlabel("direction du trait dessiné (degrés)"); ax.set_ylabel("rotation subie (degrés)")
ax.set_title("De combien chaque direction tourne-t-elle pendant l'essai ?"); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

changements = np.where(np.diff(np.sign(rotations)) != 0)[0]
print("Directions qui ne tournent pas (à 0,25° près) :", thetas[changements])

Si `ipywidgets` est installé, un curseur permet de faire tourner le vecteur à la main et de voir son image ; sinon, la cellule affiche une seule direction.

In [ ]:
def dessiner(theta=30):
    t = np.radians(theta)
    u = np.array([np.cos(t), np.sin(t)])
    cercle = np.array([np.cos(np.linspace(0, 2 * np.pi, 200)), np.sin(np.linspace(0, 2 * np.pi, 200))])
    fig, ax = plt.subplots(figsize=(4.6, 4.6))
    ax.plot(*cercle, color=LIOUAI["gris"], lw=1)
    ax.plot(*(A @ cercle), color=LIOUAI["navy"], lw=1.5)
    ax.annotate("", A @ u, (0, 0), arrowprops=dict(arrowstyle="-|>", color=LIOUAI["vert"], lw=2))
    ax.annotate("", u, (0, 0), arrowprops=dict(arrowstyle="-|>", color=LIOUAI["gris_texte"], lw=1.5))
    ax.set_title(f"u à {theta}° ; Au tourne de {angle_signe(u, A @ u):+.1f}°")
    ax.set_xlim(-2, 2); ax.set_ylim(-2, 2); ax.set_aspect("equal"); ax.grid(alpha=0.3)
    plt.show()

try:
    from ipywidgets import interact
    interact(dessiner, theta=(0, 180, 1))
except ImportError:
    dessiner(30)

### Le calcul direct : `np.linalg.eig`

In [ ]:
valeurs, vecteurs = np.linalg.eig(A)
# Selon la version de NumPy, eig renvoie toujours des nombres complexes, même s'ils sont réels.
# real_if_close les convertit en réels quand leur partie imaginaire est négligeable.
valeurs, vecteurs = np.real_if_close(valeurs), np.real_if_close(vecteurs)
for k in range(2):
    v = vecteurs[:, k]
    angle = np.degrees(np.arctan2(v[1], v[0])) % 180
    print(f"lambda = {valeurs[k]:.4f}   direction {angle:6.2f}°   A v = lambda v ? {np.allclose(A @ v, valeurs[k] * v)}")

print("\nPar le polynôme caractéristique : racines de", f"x² - {np.trace(A):.1f} x + {np.linalg.det(A):.2f}",
      "->", np.roots([1, -np.trace(A), np.linalg.det(A)]).round(4))

**Activité** : `np.linalg.eig` range les vecteurs propres **en colonnes**, et les normalise à la longueur 1. Selon la version de NumPy, il peut renvoyer des nombres complexes de partie imaginaire nulle, d'où l'appel à `np.real_if_close`. Vérifiez que les deux directions propres sont perpendiculaires. Est-ce un hasard ? Regardez la matrice $A$ : que remarquez-vous ?

Indiquez vos réponses et justifiez-les.

### Une galerie de transformations

In [ ]:
t = np.radians(30)
galerie = {
    "rotation de 30°": np.array([[np.cos(t), -np.sin(t)], [np.sin(t), np.cos(t)]]),
    "cisaillement": np.array([[1, 0.8], [0, 1]]),
    "projection sur l'axe x": np.array([[1, 0], [0, 0]]),
    "miroir (axe y)": np.array([[-1, 0], [0, 1]]),
    "étirement non symétrique": np.array([[2, 1], [0, 1]]),
}
for nom, B in galerie.items():
    w = np.real_if_close(np.linalg.eigvals(B))
    print(f"{nom:26s} valeurs propres : {np.round(w, 3)}")

**Activité** :

1. Pourquoi la rotation n'a-t-elle aucune valeur propre réelle ? Que signifie le nombre complexe affiché ?
2. Le cisaillement n'a qu'une seule direction propre, alors que la valeur propre 1 est « double ». Dessinez la situation.
3. Que signifie une valeur propre nulle pour la projection ? Faites le lien avec le module M4.

Indiquez vos réponses et justifiez-les.

### Répéter la transformation

Que se passe-t-il si l'on étire le tissu plusieurs fois de suite ? On applique $A$ dix fois à un vecteur de départ quelconque.

In [ ]:
x = np.array([0.0, 1.0])
direction_dominante = vecteurs[:, np.argmax(valeurs)]
fig, ax = plt.subplots(figsize=(5, 5))
for k in range(10):
    u = x / np.linalg.norm(x)
    ax.annotate("", u, (0, 0), arrowprops=dict(arrowstyle="-|>", color=plt.colormaps["liouai"](0.25 + 0.07 * k), lw=1.5))
    print(f"étape {k} : direction {np.degrees(np.arctan2(u[1], u[0])):6.2f}°, longueur {np.linalg.norm(x):7.2f}")
    x = A @ x
ax.plot(*np.outer([-1.2, 1.2], direction_dominante).T, "--", color=LIOUAI["vert"], label="direction propre dominante")
ax.set_xlim(-0.2, 1.2); ax.set_ylim(-0.2, 1.2); ax.set_aspect("equal"); ax.legend(loc="upper right"); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## ⑤ Interpréter

In [ ]:
k = np.argmax(valeurs)
print(f"Votre pari : direction {pari_angle_direction_principale}°, étirement {pari_etirement_maximal}")
print(f"Résultat   : direction {np.degrees(np.arctan2(vecteurs[1, k], vecteurs[0, k])) % 180:.1f}°, "
      f"étirement {valeurs[k]:.3f}")

**Activité** :

1. Rédigez pour la styliste la réponse : dans quelles directions couper le tissu, et quel allongement prévoir dans chacune ?
2. Le petit cercle est devenu une ellipse. Que représentent les deux valeurs propres sur cette ellipse ?
3. Pourquoi, en répétant la transformation, tous les vecteurs finissent-ils par s'aligner sur la même direction ? Cette idée reviendra dans tout le module.

Indiquez vos réponses et justifiez-les.

## ⑥ Critiquer et généraliser

- **Hypothèse.** On a supposé la déformation parfaitement linéaire et identique partout sur l'échantillon. Si les coefficients de $A$ sont mesurés à $\pm 0{,}02$ près, de combien la direction principale peut-elle varier ?
- **Vérifier autrement.** Retrouvez les valeurs propres par le polynôme caractéristique, et la direction principale sur la courbe d'exploration : les trois méthodes concordent-elles ?
- **Généraliser.** Écrivez une fonction `elements_propres_2x2(A)` qui calcule les valeurs propres d'une matrice $2 \times 2$ avec la formule du second degré, signale le cas où elles ne sont pas réelles, et renvoie sinon un vecteur propre normé pour chacune. Comparez-la à `np.linalg.eig` sur la galerie.

In [ ]:
def elements_propres_2x2(A):
    # À compléter : renvoie (valeurs, vecteurs) ou lève une erreur si les valeurs propres ne sont pas réelles
    ...

## Corrigé

*À n'ouvrir qu'après avoir cherché.*

In [ ]:
def elements_propres_2x2(A):
    (a, b), (c, d) = A
    tr, det = a + d, a * d - b * c
    delta = tr ** 2 - 4 * det
    if delta < 0:
        raise ValueError("Valeurs propres non réelles : la transformation fait tourner toutes les directions.")
    valeurs = [(tr + np.sqrt(delta)) / 2, (tr - np.sqrt(delta)) / 2]
    vecteurs = []
    for lam in valeurs:
        # (A - lambda I) v = 0 : on prend un vecteur orthogonal à une ligne non nulle de A - lambda I
        M = np.array(A, dtype=float) - lam * np.eye(2)
        ligne = M[0] if np.linalg.norm(M[0]) > 1e-12 else M[1]
        v = np.array([-ligne[1], ligne[0]]) if np.linalg.norm(ligne) > 1e-12 else np.array([1.0, 0.0])
        vecteurs.append(v / np.linalg.norm(v))
    return np.array(valeurs), np.column_stack(vecteurs)

for nom, B in {"tissu": A, **galerie}.items():
    try:
        w, V = elements_propres_2x2(B)
        print(f"{nom:26s} {np.round(w, 4)}   vérification A v = lambda v : {np.allclose(B @ V, V * w)}")
    except ValueError as erreur:
        print(f"{nom:26s} {erreur}")

rng = np.random.default_rng(0)
angles = []
for _ in range(1000):
    B = A + rng.uniform(-0.02, 0.02, (2, 2))
    w, V = np.linalg.eig(B)
    v = V[:, np.argmax(w.real)].real
    angles.append(np.degrees(np.arctan2(v[1], v[0])) % 180)
print(f"\nAvec des mesures à ±0,02 près, la direction principale varie entre {min(angles):.1f}° et {max(angles):.1f}°")

## Bilan — auto-évaluation

Pour chaque compétence travaillée dans ce notebook, choisissez un niveau : **0** non observé · **1** guidé · **2** autonome · **3** transfert.
La cellule enregistre votre bilan dans `progression.json`, à la racine du site.

In [ ]:
import json, datetime, pathlib

bilan = {
    "Décrire le concept": 1,
    "Avoir l'intuition du comportement de l'outil": 1,
    "Expliquer visuellement": 1,
    "Estimer la solution avant de calculer": 1,
    "Corroborer ses résultats": 1,
}   # modifiez les niveaux

ici = pathlib.Path.cwd()
racine = next((p for p in [ici, *ici.parents] if (p / "myst.yml").exists()), ici)
fichier = racine / "progression.json"
historique = json.loads(fichier.read_text(encoding="utf-8")) if fichier.exists() else []
historique.append({"notebook": "7.1-directions-invariantes", "date": datetime.date.today().isoformat(), "niveaux": bilan})
fichier.write_text(json.dumps(historique, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"Bilan enregistré ({len(historique)} entrée(s) dans {fichier.name}).")